In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

df = pd.read_csv(BASE_DIR / "data" / "cleaned_transactions.csv")
monthly_summary = pd.read_csv(BASE_DIR / "data" / "monthly_financial_summary.csv")
category_spending = pd.read_csv(BASE_DIR / "data" / "category_spending.csv")

print("Transaction data:", df.shape)
print("Monthly summary:", monthly_summary.shape)
print("Category spending:", category_spending.shape)

Transaction data: (2452, 8)
Monthly summary: (45, 4)
Category spending: (27, 2)


In [2]:
total_income = monthly_summary["Income"].sum()
total_expense = monthly_summary["Amount"].sum()
total_savings = monthly_summary["Savings"].sum()

highest_category = category_spending.iloc[0]
lowest_category = category_spending.iloc[-1]

savings_rate = (total_savings / total_income) * 100 if total_income > 0 else 0

print("Total Income:", round(total_income, 2))
print("Total Expenses:", round(total_expense, 2))
print("Total Savings:", round(total_savings, 2))
print("Savings Rate:", round(savings_rate, 2), "%")
print("Highest Spending Category:", highest_category["Category"], "-", round(highest_category["Amount"], 2))
print("Lowest Spending Category:", lowest_category["Category"], "-", round(lowest_category["Amount"], 2))

Total Income: 3042397.35
Total Expenses: 1955380.53
Total Savings: 1087016.82
Savings Rate: 35.73 %
Highest Spending Category: Money transfer - 606528.9
Lowest Spending Category: garbage disposal - 67.0


In [3]:
from google import genai

client = genai.Client()

financial_prompt = f"""
Analyze the following verified financial information.

Total Income: ₹{total_income:.2f}
Total Expenses: ₹{total_expense:.2f}
Total Savings: ₹{total_savings:.2f}
Savings Rate: {savings_rate:.2f}%

Highest Spending Category: {highest_category["Category"]}
Highest Category Amount: ₹{highest_category["Amount"]:.2f}

Lowest Spending Category: {lowest_category["Category"]}
Lowest Category Amount: ₹{lowest_category["Amount"]:.2f}

Provide:
1. A short assessment of the overall financial position.
2. Two positive observations.
3. Two areas that could be improved.
4. Three practical recommendations.

Use only the supplied figures.
Do not invent financial values.
Do not convert the figures into monthly or annual amounts.
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=financial_prompt
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Based on the financial information provided, here is the analysis:

### 1. Overall Financial Position
The financial position is strong and healthy. With a Total Income of ₹3042397.35 exceeding Total Expenses of ₹1955380.53, there is a substantial positive surplus resulting in Total Savings of ₹1087016.82 and a solid Savings Rate of 35.73%. 

### 2. Positive Observations
* **Solid Savings Performance:** Achieving Total Savings of ₹1087016.82 yields a healthy Savings Rate of 35.73% relative to the Total Income of ₹3042397.35.
* **Controlled Total Outflows:** Total Expenses of ₹1955380.53 remain well within the Total Income limit of ₹3042397.35, preventing deficit spending.

### 3. Areas That Could Be Improved
* **High Concentration in Top Spending Category:** "Money transfer" represents the highest single expense category at ₹606528.90, making up a major portion of the Total Expenses (₹1955380.53).
* **Expense Optimization Potential:** While the lowest category ("garbage disposal") is mi

In [4]:
category_analysis = category_spending.copy()

category_analysis["Percentage"] = (
    category_analysis["Amount"] / total_expense * 100
)

category_analysis = category_analysis.sort_values(
    by="Amount",
    ascending=False
)

print(category_analysis.head(10).to_string(index=False))

             Category    Amount  Percentage
       Money transfer 606528.90   31.018459
           Investment 269858.00   13.800792
       Transportation 169053.78    8.645569
            Household 161645.58    8.266707
         subscription 114587.91    5.860134
                 Food  96393.10    4.929634
Public Provident Fund  90000.00    4.602685
                Other  87025.28    4.450555
               Family  78582.20    4.018768
               Health  66252.75    3.388228


In [5]:
category_text = category_analysis.head(10).to_string(
    index=False,
    formatters={
        "Amount": "{:.2f}".format,
        "Percentage": "{:.2f}%".format
    }
)

pattern_prompt = f"""
Analyze these verified spending patterns from a personal finance dataset.

Total Expenses: ₹{total_expense:.2f}

Top Spending Categories:
{category_text}

Identify:
1. The three most important spending patterns.
2. Two categories that deserve closer attention.
3. Three practical ways to improve spending habits.

Use only the supplied data.
Do not invent values.
Do not calculate monthly or annual amounts.
Do not describe any category as monthly or annual spending.
Keep the analysis simple and practical.
"""

pattern_response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=pattern_prompt
)

print(pattern_response.text)

Based on the verified spending dataset provided, here is the analysis:

### 1. Three Most Important Spending Patterns

* **Dominance of Money Transfers:** "Money transfer" is by far the largest single expenditure, accounting for ₹6,06,528.90, which represents nearly a third (31.02%) of total spending.
* **Strong Allocation Toward Savings and Investments:** Combined financial allocations in "Investment" (13.80%, ₹2,69,858.00) and "Public Provident Fund" (4.60%, ₹90,000.00) make up 18.40% (₹3,59,858.00) of the total dataset spending.
* **Significant Core Operational Expenses:** Core living costs led by "Transportation" (8.65%, ₹1,69,053.78) and "Household" (8.27%, ₹1,61,645.58) combined account for 16.92% (₹3,30,699.36) of all expenditures.

---

### 2. Two Categories That Deserve Closer Attention

1. **Money transfer (₹6,06,528.90 / 31.02%):** Because this represents the single largest portion of overall expenses, it requires clear tracking to determine whether these are transfers betwe

In [7]:
user_question = "How can I reduce my spending based on my current spending patterns?"

question_prompt = f"""
You are a personal finance assistant.

Use the following verified financial information to answer the user's question.

Total Income: ₹{total_income:.2f}
Total Expenses: ₹{total_expense:.2f}
Total Savings: ₹{total_savings:.2f}
Savings Rate: {savings_rate:.2f}%

Top Spending Categories:
{category_text}

User Question:
{user_question}

Answer the user's question using only the supplied financial information.
Give clear, practical and personalized advice.
Do not invent financial values.
Do not calculate monthly or annual amounts.
"""

question_response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=question_prompt
)

print(question_response.text)

Based on your verified financial data, your total expenses stand at **₹1955380.53** with a savings rate of **35.73%**. 

Here is practical, personalized advice to reduce your spending by focusing on key areas in your spending patterns:

### 1. Audit "Money Transfer" Outflows
* **Current Spend:** ₹606528.90 (31.02% of total expenses)
* **Action:** This is your largest single expense category. Review where these transfers are going (e.g., informal loans, support, or unclassified transfers) to see if any of these payments can be reduced, paused, or better categorized.

### 2. Cut Down on Subscriptions
* **Current Spend:** ₹114587.91 (5.86% of total expenses)
* **Action:** Subscriptions make up a notable portion of your spending. Audit all recurring subscription services (streaming, software, memberships) and cancel any that are unused or redundant.

### 3. Optimize Transportation and Household Expenses
* **Transportation Spend:** ₹169053.78 (8.65% of total expenses)
* **Household Spend:**

In [9]:
import sys
sys.path.append(str(BASE_DIR / "src"))

from financial_advisor import answer_financial_question

user_question = "How can I reduce my spending?"

answer = answer_financial_question(
    user_question,
    total_income,
    total_expense,
    total_savings,
    savings_rate,
    category_text
)

print(answer)

Based on your provided financial breakdown (Total Expenses: **₹1955380.53**, Current Savings: **₹1087016.82**, Savings Rate: **35.73%**), here is targeted advice on how to reduce your spending:

### 1. Review Money Transfers
* **Current Spend:** ₹606528.90 (31.02% of total expenses)
* **Action:** This is your largest spending category by far. Examine where these funds are being transferred. Identify if any of these outgoing transfers can be reduced, paused, or re-negotiated.

### 2. Audit Subscriptions
* **Current Spend:** ₹114587.91 (5.86% of total expenses)
* **Action:** Go through all active subscriptions and cancel any services you do not use regularly. Consolidating or cutting unnecessary recurring subscriptions is one of the easiest ways to lower expenses.

### 3. Optimize Transportation and Household Expenses
* **Transportation:** ₹169053.78 (8.65% of total expenses)
* **Household:** ₹161645.58 (8.27% of total expenses)
* **Action:** Combined, these represent over 16% of your sp

In [10]:
import sys
import importlib

sys.path.append(str(BASE_DIR / "src"))

import financial_advisor

importlib.reload(financial_advisor)

print("Financial advisor module loaded successfully.")

Financial advisor module loaded successfully.
